In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [2]:
from transformers import AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [4]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
prompt = "tell me about cricket best player from india"    

In [6]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [8]:
outputs =model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [10]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

tell me about cricket best player from india.
Please, tell me about cricket best player from India. I am looking for information of best players of India and other team players.
The current Indian squad is very good, and many players have been performing well in international matches. However, there are some players who need to be replaced in the team.
This article will provide you with information on the top 5 best batsmen in the Indian team. The list includes Virat Kohli, KL Rahul, MS


In [11]:
from datasets import load_dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [12]:
dataset

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

In [15]:
def format_row(example):
    question = example["Context"]
    answer = example["Response"]
    example["Text"] = f"[Context] {question} [/Response] {answer}"
    return example     

In [16]:
formatted_dataset = dataset.map(format_row)

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [17]:
formatted_dataset

Dataset({
    features: ['Context', 'Response', 'Text'],
    num_rows: 3512
})

In [18]:
print(formatted_dataset[0]["Text"])

[Context] I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeling of being worthless to everyone? [/Response] If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media.  Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.

In [23]:
dataset

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

In [25]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [28]:
def tokenize_fn(example):
    tokens = tokenizer(example["Text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens  

In [30]:
tokenized = formatted_dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [31]:
from peft import LoraConfig, get_peft_model, TaskType

In [32]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [35]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 33.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [36]:
instruction_model = get_peft_model(model, lora_config)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [37]:
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)     

In [39]:
trainer = Trainer(
    model=instruction_model,
    args=args,
    train_dataset=tokenized,
)

In [40]:
trainer.train()

Step,Training Loss
20,5.421813
40,1.654287
60,1.342300
80,1.270352
100,1.351896
120,1.317030
140,1.263008
160,1.224776
180,1.346719
200,1.302175


TrainOutput(global_step=1317, training_loss=1.3215473235875437, metrics={'train_runtime': 2441.5666, 'train_samples_per_second': 4.315, 'train_steps_per_second': 0.539, 'total_flos': 3.352009798110413e+16, 'train_loss': 1.3215473235875437, 'epoch': 3.0})